# Stable Diffusion 模型结构可视化展示（基于 torchinfo）

本 Notebook 使用 `torchinfo.summary()` 对 Stable Diffusion 2.1-base 的各核心子模块进行结构可视化，
相比原始 `print(model)` 的深层缩进文本，`torchinfo` 以**表格形式**输出每一层的：
- 模块名称与类型
- 输入 / 输出 shape（同级层对齐，便于横向对比）
- 参数量（可训练 / 冻结）
- 前向传播计算量（MACs / FLOPs）

**覆盖模块：**
| 编号 | 模块 | 类 | 输入 |
|------|------|----|------|
| 一 | VAE 编码器 | `Encoder` | `[B, 3, 512, 512]` |
| 二 | VAE 解码器 | `Decoder` | `[B, 4, 64, 64]` |
| 三 | 完整 VAE | `AutoencoderKL` | `[B, 3, 512, 512]` |
| 四 | CLIP 文本编码器 | `CLIPTextModel` | `[B, 77]`（token ids）|
| 五 | U-Net 去噪网络 | `UNet2DConditionModel` | latent + timestep + text emb |

## 一、环境准备

### 1.1 安装依赖

`torchinfo` 是 `torchsummary` 的现代替代品，支持多输入、动态 shape、可配置列等特性。

In [ ]:
# !pip install torchinfo  # 安装 torchinfo，用于以表格形式展示模型各层结构、shape 及参数量

### 1.2 导入库

In [22]:
import torch                                            # 导入 PyTorch，用于构造虚拟输入张量并管理设备
from torchinfo import summary                          # 导入 torchinfo 的核心函数 summary，以表格形式打印模型结构
from diffusers import AutoencoderKL, UNet2DConditionModel  # 导入 SD 的 VAE 和 U-Net 模型类
from transformers import CLIPTextModel, CLIPTokenizer  # 导入 CLIP 文本编码器和分词器类
# modelscope 在 1.3 加载模型 cell 中按需导入，此处不重复导入

# 检查设备：优先使用 GPU，否则退回 CPU
# DEVICE 类型：str，取值 "cuda" 或 "cpu"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前使用设备：{DEVICE}")                        # 打印设备信息，便于确认运行环境

当前使用设备：cuda


### 1.3 加载模型

使用 **ModelScope `snapshot_download`** 将模型下载/读取到本地缓存，再通过本地路径加载各子模块，
避免每次运行时重复联网拉取权重。

- `snapshot_download(model_id, cache_dir)`：首次运行时联网下载，后续直接返回本地已缓存路径（str）
- `from_pretrained(local_dir, subfolder=...)`：从本地目录的对应子目录加载权重，结构与 HuggingFace 仓库一致
- `torch_dtype=torch.float16`：以半精度加载节省显存（CPU 环境自动退回 float32）
- `low_cpu_mem_usage=True`：逐层加载权重，避免峰值内存翻倍

In [23]:
from modelscope import snapshot_download                # 导入 ModelScope 的快照下载函数，首次调用时联网下载，后续直接读取本地缓存

MODEL_ID = "stabilityai/stable-diffusion-2-1-base"     # ModelScope 模型 ID（与 HuggingFace 同名），对应 SD 2.1-base 权重
MODEL_CACHE_DIR = "./model_cache/SD"                   # 本地缓存根目录：模型文件下载/缓存到此目录下的子目录中

# 根据设备选择数据类型：
# - GPU：float16 节省显存；float16 类型为 torch.dtype
# - CPU：float32（CPU 不原生支持高效 float16）；float32 类型为 torch.dtype
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# snapshot_download 返回值：str，本地模型目录的绝对路径
# 结构与 HuggingFace 仓库一致：model_dir/vae/, model_dir/unet/, model_dir/text_encoder/, ...
# 首次运行时联网下载全部文件；后续检测到缓存后直接返回路径，无需再下载
print(f"正在获取模型本地路径（首次运行时自动下载）...")
model_dir = snapshot_download(
    MODEL_ID,                                           # ModelScope 模型 ID，格式为 "组织名/模型名"
    cache_dir=MODEL_CACHE_DIR,                          # 本地缓存根目录，ModelScope 在此目录下自动管理子目录
)                                                       # model_dir: str，如 "./model_cache/SD/stabilityai/stable-diffusion-2-1-base/..."
print(f"模型本地路径：{model_dir}")                     # 打印实际本地路径，便于确认缓存位置

# ----- 加载 VAE -----
# AutoencoderKL 为变分自编码器，包含 encoder / decoder / quant_conv / post_quant_conv 四个子模块
vae = AutoencoderKL.from_pretrained(
    model_dir,                                          # 本地模型目录（snapshot_download 返回的路径）
    subfolder="vae",                                    # 子目录：SD 仓库按子目录存放各子模块权重
    torch_dtype=DTYPE,                                  # 权重数据类型：float16（GPU）或 float32（CPU）
    low_cpu_mem_usage=True,                             # 逐层加载以控制峰值内存（加载大模型时推荐开启）
).to(DEVICE)                                            # 将模型移至目标设备（GPU/CPU）
vae.eval()                                              # 切换为评估模式：关闭 Dropout，BN 使用统计值而非批次值
print("✓ VAE 加载完成")

# ----- 加载 CLIP 文本编码器 -----
# CLIPTextModel 为 Transformer 文本编码器，将 token 序列映射为 [B, 77, 1024] 的语义向量
text_encoder = CLIPTextModel.from_pretrained(
    model_dir,                                          # 本地模型目录
    subfolder="text_encoder",                           # 子目录：text_encoder 存放 CLIP 权重
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE)
text_encoder.eval()
print("✓ CLIP 文本编码器加载完成")

# ----- 加载 U-Net -----
# UNet2DConditionModel 为以文本嵌入为条件的去噪 U-Net，输出预测噪声 [B, 4, 64, 64]
unet = UNet2DConditionModel.from_pretrained(
    model_dir,                                          # 本地模型目录
    subfolder="unet",                                   # 子目录：unet 存放 U-Net 权重
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE)
unet.eval()
print("✓ U-Net 加载完成")

正在获取模型本地路径（首次运行时自动下载）...


2026-07-21 11:43:53,558 | INFO    | modelscope_hub.download | Downloading 29 files from stabilityai/stable-diffusion-2-1-base@master


Downloading:   0%|          | 0/29 [00:00<?, ?file/s]

模型本地路径：model_cache\SD\models\stabilityai--stable-diffusion-2-1-base\snapshots\master
✓ VAE 加载完成


Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

✓ CLIP 文本编码器加载完成
✓ U-Net 加载完成


## 二、VAE（变分自编码器）结构可视化

`AutoencoderKL` 包含 **Encoder** 和 **Decoder** 两条独立数据流，分别展示可以清晰看到：
- 每一层的输入/输出 shape，直观反映通道数变化（128→256→512）和空间分辨率变化（512→64）
- 同级的 `DownEncoderBlock2D` / `UpDecoderBlock2D` 模块参数量横向对比
- 每个 `ResnetBlock2D` 与 `Attention` 模块的计算占比

### 2.1 VAE 编码器（Encoder）结构

In [29]:
print("=" * 80)
print("VAE 编码器（Encoder）结构：将原始图像 [B, 3, 512, 512] → latent [B, 8, 64, 64]")
print("=" * 80)

# 说明：使用 input_data 而非 input_size 传入虚拟输入
# 原因：模型以 DTYPE（float16/float32）加载，torchinfo 的 input_size 默认创建 float32 张量，
#       与 float16 模型权重类型不匹配会抛出 RuntimeError。
#       改用 input_data 显式指定与模型一致的 dtype 和 device，避免类型冲突。
# dummy_encoder_input 形状：[batch=1, C=3, H=512, W=512]
#   - batch=1：批量大小 1，用于结构分析（不影响参数量统计）
#   - C=3    ：RGB 三通道输入
#   - H=W=512：SD 2.1-base 的标准输入分辨率
dummy_encoder_input = torch.randn(1, 3, 512, 512, dtype=DTYPE).to(DEVICE)

# summary 参数说明：
#   model        ：要分析的 nn.Module 实例，此处为 vae.encoder（单独分析编码器子模块）
#   input_data   ：实际张量（Tensor），dtype 和 device 须与模型一致；替代 input_size 避免类型不匹配
#   depth        ：展开的子模块层级深度（int），4 层足以看到 ResnetBlock 内部细节
#   col_names    ：表格显示的列名列表（list[str]），各列含义：
#                  - "input_size"  ：该层的输入 shape，如 [1, 3, 512, 512]
#                  - "output_size" ：该层的输出 shape，如 [1, 128, 512, 512]
#                  - "num_params"  ：该层的参数量（int），含可训练和冻结
#                  - "trainable"   ：该层是否有可训练参数（bool）
#   col_width    ：每列宽度（int, 单位字符），防止 shape 信息被截断
#   row_settings ：行显示设置（set[str]），"var_names" 表示显示变量名/模块路径
#   verbose      ：0=只打印摘要行，1=打印完整表格（int），此处选 1 显示全部层
_ = summary(
    vae.encoder,                                        # 单独分析 VAE 编码器子模块（nn.Module 类型）
    input_data=dummy_encoder_input,                     # 虚拟输入张量，dtype=DTYPE，device=DEVICE，形状 [1, 3, 512, 512]
    depth=10,                                           # 展开到最深层（depth=10 覆盖所有实际层级）：Block → ResnetBlock → Conv2d/GroupNorm/SiLU 等原语层
    col_names=["input_size", "output_size", "num_params", "trainable"],  # 显示输入shape/输出shape/参数量/可训练标志
    col_width=30,                                       # 列宽 30 字符，避免较长 shape 字符串被截断
    row_settings={"var_names"},                         # 在行首显示模块路径名（如 down_blocks.0.resnets.0）
    verbose=1,                                          # 输出详细表格（0 为仅摘要）
)  # _ = 赋值：阻止 Jupyter 自动 repr() 显示 ModelStatistics，避免输出重复两次

VAE 编码器（Encoder）结构：将原始图像 [B, 3, 512, 512] → latent [B, 8, 64, 64]
Layer (type (var_name))                            Input Shape                    Output Shape                   Param #                        Trainable
Encoder (Encoder)                                  [1, 3, 512, 512]               [1, 8, 64, 64]                 --                             True
├─Conv2d (conv_in)                                 [1, 3, 512, 512]               [1, 128, 512, 512]             3,584                          True
├─ModuleList (down_blocks)                         --                             --                             --                             True
│    └─DownEncoderBlock2D (0)                      [1, 128, 512, 512]             [1, 128, 256, 256]             --                             True
│    │    └─ModuleList (resnets)                   --                             --                             --                             True
│    │    │    └─ResnetBlock2D (0) 

### 2.2 VAE 解码器（Decoder）结构

In [30]:
print("=" * 80)
print("VAE 解码器（Decoder）结构：将 latent [B, 4, 64, 64] → 还原图像 [B, 3, 512, 512]")
print("=" * 80)

# 解码器输入为 4 通道 latent（重参数化采样后的结果）
# dummy_decoder_input 形状：[batch=1, C=4, H=64, W=64]
#   - C=4    ：latent 通道数（VAE 编码 + 重参数化后为 4 通道）
#   - H=W=64 ：latent 空间分辨率（原始 512×512 经 VAE 编码缩小 8 倍）
# dtype=DTYPE, device=DEVICE 须与模型一致，避免 float32 vs float16 类型冲突
dummy_decoder_input = torch.randn(1, 4, 64, 64, dtype=DTYPE).to(DEVICE)

_ = summary(
    vae.decoder,                                        # 单独分析 VAE 解码器子模块（nn.Module 类型）
    input_data=dummy_decoder_input,                     # 虚拟 latent 张量，形状 [1, 4, 64, 64]，dtype=DTYPE
    depth=10,                                           # 展开到最深层（depth=10 覆盖所有实际层级），与编码器保持一致以便对比
    col_names=["input_size", "output_size", "num_params", "trainable"],  # 同编码器列设置
    col_width=30,                                       # 列宽 30 字符
    row_settings={"var_names"},                         # 显示模块路径（如 up_blocks.0.resnets.0）
    verbose=1,                                          # 输出详细表格
)  # _ = 赋值：阻止 Jupyter 自动 repr() 显示 ModelStatistics，避免输出重复两次

VAE 解码器（Decoder）结构：将 latent [B, 4, 64, 64] → 还原图像 [B, 3, 512, 512]
Layer (type (var_name))                            Input Shape                    Output Shape                   Param #                        Trainable
Decoder (Decoder)                                  [1, 4, 64, 64]                 [1, 3, 512, 512]               --                             True
├─Conv2d (conv_in)                                 [1, 4, 64, 64]                 [1, 512, 64, 64]               18,944                         True
├─UNetMidBlock2D (mid_block)                       [1, 512, 64, 64]               [1, 512, 64, 64]               --                             True
│    └─ModuleList (resnets)                        --                             --                             (recursive)                    True
│    │    └─ResnetBlock2D (0)                      [1, 512, 64, 64]               [1, 512, 64, 64]               --                             True
│    │    │    └─GroupNorm (norm1)

### 2.3 完整 VAE（AutoencoderKL）— 仅统计顶层参数分布

完整 VAE 的前向传播需要经过 `encode → reparameterize → decode` 三步，
此处用 `depth=1` 仅展示顶层子模块参数量汇总（不做完整前向，避免 VAE 返回值类型不兼容问题）。

In [31]:
print("=" * 80)
print("完整 VAE（AutoencoderKL）顶层子模块参数量汇总（depth=1）")
print("=" * 80)

# depth=1 只展示直接子模块（encoder / decoder / quant_conv / post_quant_conv），不深入内部
# 此处不传入 input_size，仅统计参数量（避免完整前向传播时 VAEOutput 类型导致 torchinfo 解析异常）
# col_names 去掉 input_size/output_size，只看参数量分布
_ = summary(
    vae,                                                # 完整 VAE 模型（AutoencoderKL 类型，nn.Module 子类）
    depth=1,                                            # 仅展开一层：直接子模块（encoder/decoder/quant_conv等）
    col_names=["num_params", "trainable"],              # 只显示参数量和可训练标志（不做前向，无 shape 信息）
    col_width=25,                                       # 列宽 25 字符
    row_settings={"var_names"},                         # 显示子模块名称路径
    verbose=1,                                          # 详细输出
)  # _ = 赋值：阻止 Jupyter 自动 repr() 显示 ModelStatistics，避免输出重复两次

完整 VAE（AutoencoderKL）顶层子模块参数量汇总（depth=1）
Layer (type (var_name))                                 Param #                   Trainable
AutoencoderKL (AutoencoderKL)                           --                        True
├─Encoder (encoder)                                     34,163,592                True
├─Decoder (decoder)                                     49,490,179                True
├─Conv2d (quant_conv)                                   72                        True
├─Conv2d (post_quant_conv)                              20                        True
Total params: 83,653,863
Trainable params: 83,653,863
Non-trainable params: 0


## 三、CLIP 文本编码器（CLIPTextModel）结构可视化

`CLIPTextModel` 接收的输入是**整数 token id 张量**（形状 `[B, 77]`），而非浮点特征图。
`torchinfo` 不能用 `input_size` 自动构造（默认生成 float 零张量），
需改用 `input_data` 显式传入正确数据类型的张量。

输出为 `[B, 77, 1024]` 的文本语义序列，随后作为 Cross-Attention 的 K/V 注入 U-Net。

### 3.1 CLIPTextTransformer 主体结构（depth=10，展开到最深层）

In [32]:
print("=" * 80)
print("CLIP 文本编码器（CLIPTextModel）结构：token ids [B,77] → 文本嵌入 [B, 77, 1024]")
print("=" * 80)

# 构造虚拟 token id 张量：dtype 必须为 torch.long（int64），因为 Embedding 层要求整数索引
# dummy_token_ids 形状：[batch=1, seq_len=77]
#   - batch=1：批量大小 1
#   - seq_len=77：CLIP 文本编码器最大序列长度，超出时截断
# 填充值 49406 为 BOS（序列开始）token id，模拟有效输入（也可全用 0，即 pad token）
dummy_token_ids = torch.ones(1, 77, dtype=torch.long).to(DEVICE) * 49406

# 使用 input_data 而非 input_size：input_data 允许直接传入真实张量（含正确 dtype）
# torchinfo 会以该张量做一次实际前向传播，得到每层真实的输入/输出 shape
_ = summary(
    text_encoder,                                       # CLIPTextModel 实例（nn.Module 子类）
    input_data=dummy_token_ids,                         # 虚拟 token id 张量，形状 [1, 77]，dtype=torch.long
    depth=10,                                           # 展开到最深层（depth=10 覆盖所有实际层级）：Embeddings → CLIPEncoderLayer → Attn/MLP → Linear/LayerNorm
    col_names=["input_size", "output_size", "num_params", "trainable"],  # 显示四列信息
    col_width=35,                                       # 列宽 35，给 Transformer shape 留足空间
    row_settings={"var_names"},                         # 显示模块路径名（如 text_model.encoder.layers.0.self_attn）
    verbose=1,                                          # 详细表格输出
)  # _ = 赋值：阻止 Jupyter 自动 repr() 显示 ModelStatistics，避免输出重复两次

CLIP 文本编码器（CLIPTextModel）结构：token ids [B,77] → 文本嵌入 [B, 77, 1024]
Layer (type (var_name))                                      Input Shape                         Output Shape                        Param #                             Trainable
CLIPTextModel (CLIPTextModel)                                [1, 77]                             [1, 1024]                           --                                  True
├─CLIPTextEmbeddings (embeddings)                            --                                  [1, 77, 1024]                       --                                  True
│    └─Embedding (token_embedding)                           [1, 77]                             [1, 77, 1024]                       50,593,792                          True
│    └─Embedding (position_embedding)                        [1, 77]                             [1, 77, 1024]                       78,848                              True
├─CLIPEncoder (encoder)                                    

### 3.2 单个 CLIPEncoderLayer 内部结构放大（depth=10）

`CLIPTextModel` 共 23 层 `CLIPEncoderLayer`，结构完全相同。
单独对第 0 层展开到 `depth=10` 可看到 Attention 的 Q/K/V 投影和 MLP 的 fc1/fc2 逐层 shape。

In [33]:
print("=" * 80)
print("单个 CLIPEncoderLayer（第 0 层）内部结构：self_attn / layer_norm / mlp 各子层 shape")
print("=" * 80)

# 动态查找第 0 个 CLIPEncoderLayer，不依赖版本敏感的属性路径
# 不同版本 Transformers 的 CLIPTextModel 内部层级不同：
#   - 旧版：text_encoder.text_model.encoder.layers[0]
#   - 新版：text_encoder.encoder.layers[0]（无 text_model 中间层）
# 使用 named_modules() 遍历所有子模块，按类名匹配，兼容所有版本
from transformers.models.clip.modeling_clip import CLIPEncoderLayer  # 导入 CLIPEncoderLayer 类用于 isinstance 判断

encoder_layer_0 = None                                  # 初始化为 None，找到后赋值
for layer_name, module in text_encoder.named_modules():  # 遍历所有子模块（含深层嵌套），layer_name: str，module: nn.Module
    if isinstance(module, CLIPEncoderLayer):            # 判断是否为 CLIPEncoderLayer 类型（第一个匹配即为第 0 层）
        encoder_layer_0 = module                        # 赋值第 0 个 CLIPEncoderLayer 实例
        print(f"找到 CLIPEncoderLayer，模块路径：{layer_name}")  # 打印实际路径，便于确认版本兼容性
        break                                           # 找到第一个即退出循环

# CLIPEncoderLayer.forward 签名随 Transformers 版本变化：
#   - 旧版（<4.40）：forward(hidden_states, attention_mask, causal_attention_mask)  3个参数
#   - 新版（≥4.40）：forward(hidden_states, attention_mask)                         2个参数（causal_mask 已内化）
# 本环境为新版，只需传 2 个参数；通过 input_data 列表按位置传入。
#
# ① dummy_hidden（hidden_states）：形状 [B=1, seq_len=77, hidden_dim=1024]
#    - 模拟 CLIPTextEmbeddings 的输出，经过 token_embedding + position_embedding 求和后的浮点张量
dummy_hidden = torch.randn(1, 77, 1024, dtype=DTYPE).to(DEVICE)

# ② dummy_attn_mask（attention_mask）：形状 [B=1, 1, seq_len=77, seq_len=77]
#    - 扩展后的 padding mask（additive mask）：0 表示该位置参与注意力，-inf 表示被遮蔽
#    - 后两维 [77, 77] 对应 Q×K^T 得分矩阵尺寸（seq_len × seq_len）
#    - 全零即不遮蔽任何位置（模拟无 padding 的有效序列）
dummy_attn_mask = torch.zeros(1, 1, 77, 77, dtype=DTYPE).to(DEVICE)

_ = summary(
    encoder_layer_0,                                    # 单个 CLIPEncoderLayer 实例（动态查找，版本兼容）
    input_data=[dummy_hidden, dummy_attn_mask],         # 新版只需 2 个位置参数：hidden_states + attention_mask
    depth=10,                                           # 展开到最深层（depth=10）：Linear / LayerNorm / GELUActivation
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=35,
    row_settings={"var_names"},
    verbose=1,
)  # _ = 赋值：阻止 Jupyter 自动 repr() 显示 ModelStatistics，避免输出重复两次

单个 CLIPEncoderLayer（第 0 层）内部结构：self_attn / layer_norm / mlp 各子层 shape
找到 CLIPEncoderLayer，模块路径：encoder.layers.0
Layer (type (var_name))                       Input Shape                         Output Shape                        Param #                             Trainable
CLIPEncoderLayer (CLIPEncoderLayer)           [1, 77, 1024]                       [1, 77, 1024]                       --                                  True
├─LayerNorm (layer_norm1)                     [1, 77, 1024]                       [1, 77, 1024]                       2,048                               True
├─CLIPAttention (self_attn)                   --                                  [1, 77, 1024]                       --                                  True
│    └─Linear (q_proj)                        [1, 77, 1024]                       [1, 77, 1024]                       1,049,600                           True
│    └─Linear (k_proj)                        [1, 77, 1024]                       [1, 77

## 四、U-Net（UNet2DConditionModel）结构可视化

U-Net 是多输入模型，前向传播需要同时传入三个张量：

| 参数名 | 含义 | 形状 | dtype |
|--------|------|------|-------|
| `sample` | 带噪 latent | `[B, 4, 64, 64]` | float16/32 |
| `timestep` | 当前扩散时间步 | `[B]` 或标量 | long / float |
| `encoder_hidden_states` | CLIP 文本嵌入 | `[B, 77, 1024]` | float16/32 |

`torchinfo` 通过 `input_data` 以列表形式传入多个位置参数，对应 `forward` 的签名顺序。

### 4.1 U-Net 顶层结构概览（depth=2）

`depth=2` 展示到 `down_blocks / mid_block / up_blocks` 一级，清晰看到编解码器各阶段的参数量占比。

In [34]:
print("=" * 80)
print("U-Net（UNet2DConditionModel）顶层结构：depth=2，展示各大模块参数量分布")
print("=" * 80)

# 构造 U-Net 前向传播所需的三个虚拟输入张量
# ① 带噪 latent：形状 [batch=1, C=4, H=64, W=64]
#    - batch=1：批量大小（分析结构时取 1）
#    - C=4：latent 通道数（VAE 编码 + 重参数化后为 4 通道）
#    - H=W=64：latent 空间分辨率（原始 512×512 经 VAE 编码后缩小 8 倍）
dummy_latent = torch.randn(1, 4, 64, 64, dtype=DTYPE).to(DEVICE)

# ② 时间步：形状 [1]，值为 999（扩散链末端，t=T 对应最大噪声步）
#    - dtype=torch.long：U-Net forward 接受整数时间步后内部转换为正弦嵌入
dummy_timestep = torch.tensor([999], dtype=torch.long).to(DEVICE)

# ③ CLIP 文本嵌入：形状 [batch=1, seq_len=77, hidden_dim=1024]
#    - 模拟 CLIPTextModel 输出的语义向量序列，作为 Cross-Attention 的 K/V 来源
dummy_text_emb = torch.randn(1, 77, 1024, dtype=DTYPE).to(DEVICE)

# input_data 传入列表：列表中的元素按 forward 参数顺序排列
# UNet2DConditionModel.forward 签名（简化）：forward(sample, timestep, encoder_hidden_states, ...)
_ = summary(
    unet,                                               # UNet2DConditionModel 实例（nn.Module 子类）
    input_data=[dummy_latent, dummy_timestep, dummy_text_emb],  # 三个虚拟输入，按 forward 参数顺序
    depth=2,                                            # 展开 2 层：顶级子模块（conv_in/down_blocks/mid_block/up_blocks/conv_out）
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=35,
    row_settings={"var_names"},
    verbose=1,
)  # _ = 赋值：阻止 Jupyter 自动 repr() 显示 ModelStatistics，避免输出重复两次

U-Net（UNet2DConditionModel）顶层结构：depth=2，展示各大模块参数量分布
Layer (type (var_name))                                           Input Shape                         Output Shape                        Param #                             Trainable
UNet2DConditionModel (UNet2DConditionModel)                       [1, 4, 64, 64]                      [1, 4, 64, 64]                      --                                  True
├─Timesteps (time_proj)                                           [1]                                 [1, 320]                            --                                  --
├─TimestepEmbedding (time_embedding)                              [1, 320]                            [1, 1280]                           --                                  True
│    └─Linear (linear_1)                                          [1, 320]                            [1, 1280]                           410,880                             True
│    └─SiLU (act)                                 

### 4.2 U-Net 下采样路径（down_blocks）详细展开（depth=10）

单独分析 `down_blocks` 可清晰看到 4 个下采样块内部每个 `ResnetBlock2D` 和 `Transformer2DModel` 的 shape 变化。

In [35]:
print("=" * 80)
print("U-Net 下采样路径（down_blocks）详细展开：通道 320→640→1280→1280，空间 64→8")
print("=" * 80)

# 以第 0 个下采样块（CrossAttnDownBlock2D，通道 320，空间 64→32）为例做深度展开
# 该块的 forward 需要三个参数：hidden_states、temb（时间步嵌入）、encoder_hidden_states
# - hidden_states：形状 [1, 320, 64, 64]，经过 conv_in 后的特征图
# - temb：形状 [1, 1280]，TimeStepEmbedding 输出的时间步嵌入向量
# - encoder_hidden_states：形状 [1, 77, 1024]，文本嵌入（用于 Cross-Attention）
dummy_feat_320 = torch.randn(1, 320, 64, 64, dtype=DTYPE).to(DEVICE)   # conv_in 输出的特征图，[1, 320, 64, 64]
dummy_temb = torch.randn(1, 1280, dtype=DTYPE).to(DEVICE)               # 时间步嵌入向量，[1, 1280]
dummy_cross = torch.randn(1, 77, 1024, dtype=DTYPE).to(DEVICE)          # 文本嵌入，[1, 77, 1024]

print("\n--- down_blocks[0]：CrossAttnDownBlock2D（通道 320，空间 64→32）---")
# input_data 传入列表：对应 CrossAttnDownBlock2D.forward(hidden_states, temb, encoder_hidden_states)
_ = summary(
    unet.down_blocks[0],                                # 第 0 个下采样块（CrossAttnDownBlock2D 类型）
    input_data=[dummy_feat_320, dummy_temb, dummy_cross],  # 三个输入：特征图 / 时间步嵌入 / 文本嵌入
    depth=10,                                           # 展开到最深层（depth=10 覆盖所有实际层级）：Block → Transformer → BasicTransformerBlock → Attn/FFN → Linear
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=35,
    row_settings={"var_names"},
    verbose=1,
)  # _ = 赋值：阻止 Jupyter 自动 repr() 显示 ModelStatistics，避免输出重复两次

U-Net 下采样路径（down_blocks）详细展开：通道 320→640→1280→1280，空间 64→8

--- down_blocks[0]：CrossAttnDownBlock2D（通道 320，空间 64→32）---
Layer (type (var_name))                                 Input Shape                         Output Shape                        Param #                             Trainable
CrossAttnDownBlock2D (CrossAttnDownBlock2D)             [1, 320, 64, 64]                    [1, 320, 32, 32]                    --                                  True
├─ModuleList (resnets)                                  --                                  --                                  (recursive)                         True
│    └─ResnetBlock2D (0)                                [1, 320, 64, 64]                    [1, 320, 64, 64]                    --                                  True
│    │    └─GroupNorm (norm1)                           [1, 320, 64, 64]                    [1, 320, 64, 64]                    640                                 True
│    │    └─SiLU (nonlinearity)

### 4.3 U-Net 瓶颈层（mid_block）结构（depth=10）

`mid_block`（`UNetMidBlock2DCrossAttn`）工作在最小分辨率 8×8，
包含 ResNet → CrossAttn Transformer → ResNet 结构，是文本条件注入最深的位置。

In [36]:
print("=" * 80)
print("U-Net 瓶颈层（UNetMidBlock2DCrossAttn）：1280 通道，最小分辨率 8×8")
print("=" * 80)

# mid_block 工作在下采样路径末端，输入特征图分辨率为 8×8，通道数为 1280
# mid_block.forward(hidden_states, temb, encoder_hidden_states) 与 down_block 接口相同
dummy_feat_1280_8 = torch.randn(1, 1280, 8, 8, dtype=DTYPE).to(DEVICE)  # 最小分辨率特征图 [1, 1280, 8, 8]
# dummy_temb：复用之前定义的 [1, 1280] 时间步嵌入
# dummy_cross：复用之前定义的 [1, 77, 1024] 文本嵌入

_ = summary(
    unet.mid_block,                                     # U-Net 瓶颈层（UNetMidBlock2DCrossAttn 类型）
    input_data=[dummy_feat_1280_8, dummy_temb, dummy_cross],  # 最小分辨率特征图 / 时间步嵌入 / 文本嵌入
    depth=10,                                           # 展开到最深层（depth=10 覆盖所有实际层级），观察 Transformer 内部 attn/ff 直至 Linear 原语层
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=35,
    row_settings={"var_names"},
    verbose=1,
)  # _ = 赋值：阻止 Jupyter 自动 repr() 显示 ModelStatistics，避免输出重复两次

U-Net 瓶颈层（UNetMidBlock2DCrossAttn）：1280 通道，最小分辨率 8×8
Layer (type (var_name))                                 Input Shape                         Output Shape                        Param #                             Trainable
UNetMidBlock2DCrossAttn (UNetMidBlock2DCrossAttn)       [1, 1280, 8, 8]                     [1, 1280, 8, 8]                     --                                  True
├─ModuleList (resnets)                                  --                                  --                                  (recursive)                         True
│    └─ResnetBlock2D (0)                                [1, 1280, 8, 8]                     [1, 1280, 8, 8]                     --                                  True
│    │    └─GroupNorm (norm1)                           [1, 1280, 8, 8]                     [1, 1280, 8, 8]                     2,560                               True
│    │    └─SiLU (nonlinearity)                         [1, 1280, 8, 8]                     [1, 1

## 五、torchinfo 关键参数实验对比

本节通过修改 `depth`、`col_names`、`row_settings` 等参数，
直观理解各参数对展示效果的影响，以 VAE 编码器为实验对象。

### 5.1 depth 参数对比：depth=1 vs depth=2 vs depth=4

`depth` 控制展开的层级深度，越大越详细但输出行数也越多。

In [37]:
def show_vae_encoder_with_depth(depth: int) -> None:
    """
    以指定 depth 展示 VAE 编码器结构，用于对比不同深度下的信息量差异。

    参数:
        depth (int): torchinfo summary 的展开层级深度。
                     - 1：仅显示直接子模块（conv_in / down_blocks / mid_block / conv_out）
                     - 2：展开到 Block 层（DownEncoderBlock2D / UNetMidBlock2D）
                     - 4：展开到 Conv/Norm 原语层（ResnetBlock 内部细节）
    """
    print(f"\n{'=' * 70}")
    print(f"VAE 编码器 — depth={depth}")
    print(f"{'=' * 70}")
    # input_data 须在函数内部构造（每次调用使用独立张量），dtype=DTYPE 与模型权重类型保持一致
    # dummy_input 形状 [1, 3, 512, 512]：batch=1, C=3, H=W=512
    dummy_input = torch.randn(1, 3, 512, 512, dtype=DTYPE).to(DEVICE)
    _ = summary(
        vae.encoder,                                    # VAE 编码器子模块
        input_data=dummy_input,                         # 虚拟输入张量，dtype=DTYPE，device=DEVICE，形状 [1, 3, 512, 512]
        depth=depth,                                    # 展开层级深度（由参数控制）
        col_names=["output_size", "num_params"],        # 简化列（仅输出 shape 和参数量）
        col_width=30,                                   # 列宽 30 字符
        verbose=0,                                      # verbose=0：只打印摘要行（总参数量），不输出完整表格
    )


# 依次用 depth=1,2,4 展示，观察展开层级对信息粒度的影响
for d in [1, 2, 4]:                                     # 遍历三种深度值
    show_vae_encoder_with_depth(d)                      # 调用展示函数，打印对应深度的 summary


VAE 编码器 — depth=1

VAE 编码器 — depth=2

VAE 编码器 — depth=4


### 5.2 col_names 参数：显示 MACs（计算量）

增加 `"mult_adds"` 列可展示每层的乘加运算次数（MACs），
方便比较 ResnetBlock（卷积密集）vs Attention（QKV 线性层）的计算开销。

In [38]:
print("=" * 80)
print("VAE 编码器 — 显示 MACs（乘加运算次数），对比 Conv 与 Attention 的计算开销")
print("=" * 80)

# col_names 中加入 "mult_adds"（乘加运算次数，即 MACs，Multiply-Accumulate Operations）
# MACs 数值越大，该层计算越重（卷积 MACs ∝ 输出尺寸 × 卷积核大小 × 输入通道）
# dummy_macs_input 形状 [1, 3, 512, 512]，dtype=DTYPE 与模型一致，避免 dtype 冲突
dummy_macs_input = torch.randn(1, 3, 512, 512, dtype=DTYPE).to(DEVICE)

_ = summary(
    vae.encoder,                                        # VAE 编码器子模块
    input_data=dummy_macs_input,                        # 虚拟输入张量，dtype=DTYPE，device=DEVICE，形状 [1, 3, 512, 512]
    depth=10,                                           # 展开到最深层（depth=10），让每个 Conv2d/Linear 的 MACs 都单独列出
    col_names=[
        "output_size",                                  # 该层输出 shape，如 [1, 128, 512, 512]
        "num_params",                                   # 该层参数量（int），如 147456
        "mult_adds",                                    # 该层 MACs 数（int），反映计算量大小
    ],
    col_width=28,                                       # 列宽 28 字符
    row_settings={"var_names"},                         # 显示模块路径名
    verbose=1,                                          # 详细表格输出
)  # _ = 赋值：阻止 Jupyter 自动 repr() 显示 ModelStatistics，避免输出重复两次

VAE 编码器 — 显示 MACs（乘加运算次数），对比 Conv 与 Attention 的计算开销
Layer (type (var_name))                            Output Shape                 Param #                      Mult-Adds
Encoder (Encoder)                                  [1, 8, 64, 64]               --                           --
├─Conv2d (conv_in)                                 [1, 128, 512, 512]           3,584                        939,524,096
├─ModuleList (down_blocks)                         --                           --                           --
│    └─DownEncoderBlock2D (0)                      [1, 128, 256, 256]           --                           --
│    │    └─ModuleList (resnets)                   --                           --                           --
│    │    │    └─ResnetBlock2D (0)                 [1, 128, 512, 512]           --                           --
│    │    │    │    └─GroupNorm (norm1)            [1, 128, 512, 512]           256                          256
│    │    │    │    └─SiLU (nonline

### 5.3 各模块参数量汇总对比表

用代码提取各子模块的总参数量，生成汇总对比，直观感受 U-Net >> VAE >> CLIP 的规模关系。

In [39]:
def count_params(module) -> int:
    """
    统计 nn.Module 中所有参数的总元素数量（包括可训练和冻结参数）。

    参数:
        module (nn.Module): 任意 PyTorch 模型或子模块实例。

    返回:
        int: 模型所有参数张量的元素总数（即 sum(p.numel() for p in module.parameters())）。
             例如一个 Linear(1024, 1024) 层返回 1024*1024 + 1024 = 1049600。
    """
    return sum(p.numel() for p in module.parameters())  # 遍历所有参数张量，累加每个张量的元素数（numel = number of elements）


# 构造各子模块的参数量统计字典
# 键（str）：模块描述名称；值（int）：该模块的总参数量
modules_info = {
    "VAE - 完整（AutoencoderKL）":    count_params(vae),           # 完整 VAE（含 encoder + decoder + quant_conv）
    "VAE - Encoder":                   count_params(vae.encoder),   # 仅编码器部分
    "VAE - Decoder":                   count_params(vae.decoder),   # 仅解码器部分
    "CLIP 文本编码器（CLIPTextModel）": count_params(text_encoder),  # 完整 CLIP 文本 Transformer
    "U-Net（UNet2DConditionModel）":   count_params(unet),          # 完整 U-Net（含 down/mid/up blocks）
}

print(f"\n{'模块名称':<35} {'参数量':>15} {'参数量（百万M）':>15}")
print("-" * 70)
for name, n_params in modules_info.items():                             # 遍历模块信息字典，逐行打印
    # n_params / 1e6：将参数量转换为百万（M）单位，便于直观比较规模
    print(f"{name:<35} {n_params:>15,} {n_params / 1e6:>14.1f}M")


模块名称                                            参数量        参数量（百万M）
----------------------------------------------------------------------
VAE - 完整（AutoencoderKL）                  83,653,863           83.7M
VAE - Encoder                            34,163,592           34.2M
VAE - Decoder                            49,490,179           49.5M
CLIP 文本编码器（CLIPTextModel）               340,387,840          340.4M
U-Net（UNet2DConditionModel）             865,910,724          865.9M
